# Segmentação de Clientes (Clustering)

Este notebook foca no **ETL, Limpeza e Pré-processamento** dos dados para preparar a base para modelagem.

### O que é Clustering?
O clustering é uma técnica usada em aprendizado não supervisionado para agrupar pontos de dados semelhantes com base em suas características intrínsecas ou similaridades. Seu objetivo é identificar padrões ou estruturas nos dados sem conhecimento prévio dos agrupamentos ou rótulos.

No clustering, os pontos de dados são agrupados em clusters com base em sua proximidade uns aos outros no espaço de características. O objetivo é maximizar a similaridade dentro dos clusters e minimizar a similaridade entre clusters diferentes.

### 1. Importando Módulos e Carregando Dados

In [49]:
import pandas as pd 
import os
from datetime import datetime

# Carregando os datasets
df_transacoes = pd.read_csv('../data/df_transacoes.csv')
df_produtos = pd.read_csv('../data/df_produtos.csv')
df_vendas = pd.read_csv('../data/df_vendas.csv')
df_clientes = pd.read_csv('../data/df_clientes.csv')

## 2. ETL & Limpeza

Nesta etapa, tratamos os tipos de dados e verificamos a integridade da base.

### 2.1 Conversão de Tipos (Datas)
Garantir que colunas temporais sejam tratadas como objetos `datetime` do pandas.

In [50]:
# Convertendo colunas de data
df_vendas['data_venda'] = pd.to_datetime(df_vendas['data_venda'])
df_clientes['data_nascimento'] = pd.to_datetime(df_clientes['data_nascimento'])
df_clientes['data_cadastro'] = pd.to_datetime(df_clientes['data_cadastro'])

print("Tipos de dados após conversão:")
print(df_vendas.dtypes[['data_venda']])

Tipos de dados após conversão:
data_venda    datetime64[ns]
dtype: object


### 2.2 Verificação de Valores Nulos e Duplicatas

In [51]:
datasets = {
    "Vendas": df_vendas, 
    "Clientes": df_clientes, 
    "Produtos": df_produtos, 
    "Transações": df_transacoes
}

for nome, df in datasets.items():
    nulos = df.isnull().sum().sum()
    duplicatas = df.duplicated().sum()
    print(f"{nome:12} | Nulos: {nulos} | Duplicatas: {duplicatas}")

Vendas       | Nulos: 0 | Duplicatas: 0
Clientes     | Nulos: 0 | Duplicatas: 0
Produtos     | Nulos: 0 | Duplicatas: 0
Transações   | Nulos: 0 | Duplicatas: 0


## 3. Pré-processamento & Engenharia de Atributos

Criação de novas variáveis para facilitar a análise de agrupamento.

### 1. Mensuração do Sucesso Financeiro

O que faz: Consolida o faturamento bruto de cada transação.

Por que é importante: É a base para o cálculo do LTV (Lifetime Value). Sem isso, não sabemos quem são os clientes "Baleias" (que gastam muito) e os clientes de baixo valor.

In [52]:
# 1. Valor Total da Venda (Quantidade * Preço)
df_vendas['valor_total'] = df_vendas['qtd_vendida'] * df_vendas['preco_praticado']

### 2 e 3. Enriquecimento de Contexto (Denormalização)

O que faz: Une as tabelas isoladas de Produtos e Clientes em um único "Dataset Mestre".

Por que é importante: Dados isolados são apenas registros. Cruzá-los permite responder: "Quem compra o quê?". É aqui que transformamos um ID anônimo em um perfil real com localização e preferências.

In [53]:
# 2. Trazer informações de produto para tabela de vendas
df_registro = pd.merge(df_vendas, df_produtos, on='produto_id', how='left')

# 3. Trazer informações de cliente para tabela de vendas    
df_registro = pd.merge(df_registro, df_clientes, on='cliente_id', how='left')

### 4. Análise de Elasticidade e Margem

O que faz: Calcula o desconto real (em R$ e %) aplicado em cada item.

Por que é importante: Identifica a saúde financeira das vendas. Se a diferença for sempre alta, o cliente pode estar viciado em promoções, o que corrói a margem de lucro a longo prazo.

In [54]:
# 4. Calcule a diferença entre o preco_base e o preco_praticado em percentual ou valor absoluto
df_registro['diferenca_preco'] = df_registro['preco_base'] - df_registro['preco_praticado']
df_registro['diferenca_preco_percentual'] = (df_registro['diferenca_preco'] / df_registro['preco_base']).fillna(0)


### 5 e 6. Ciclo de Vida e Maturidade

O que faz: Determina a idade do cliente e sua "idade" dentro da loja (tenure).

Por que é importante: Permite segmentação geracional e identifica o momento do cliente. Um cliente antigo que compra pouco tem um perfil diferente de um novato que acabou de se cadastrar e já fez uma compra grande.

In [55]:
# 5. Calcule a idade do cliente no momento da venda
df_registro['idade_cliente'] = (df_registro['data_venda'] - df_registro['data_nascimento']).dt.days // 365

# 6. Calcule o tempo desde o cadastro do cliente até a venda
df_registro['tempo_cadastro_venda'] = (df_registro['data_venda'] - df_registro['data_cadastro']).dt.days

### 7. Ritmo e Recência

que faz: Mede o intervalo de tempo entre as interações do cliente.

Por que é importante: É o principal indicador de Engajamento. Intervalos crescentes são um sinal de alerta (churn) iminente, permitindo ações preventivas de marketing.

In [56]:
# 7. calcular o dias desde a ultima compra do cliente se for a primeira compra, deixar como NaN ou 0    
dt_compras = df_registro.sort_values(by=['cliente_id', 'data_venda'])
df_registro['dias_desde_ultima_compra'] = df_registro.groupby('cliente_id')['data_venda'].diff().dt.days.fillna(0)

### 8. Identificação de Preferência (Persona)

O que faz: Extrai o "carro-chefe" de cada cliente usando a moda estatística.

Por que é importante: Essencial para a Personalização. Se o modelo descobre que sua categoria principal é "Escrita", o marketing não deve gastar energia enviando ofertas de "Móveis de Escritório" para você.

In [57]:
# 8. Qual item mais comprado por cliente ou qual embalagem mais comprada por cliente (ser for lapis, qual embalagem mais comprada : 12 unidades, 24 unidades, etc)
df_registro['categoria_mais_comprada'] = df_registro.groupby('cliente_id')['categoria'].transform(lambda x: x.mode()[0])

### 9. Sensibilidade Promocional

O que faz: Cria uma flag binária para compras com desconto relevante (>10%).

Por que é importante: Categoriza os clientes pelo gatilho de compra. Alguns clientes buscam qualidade/marca, outros buscam apenas a oportunidade do preço baixo.

In [58]:
# 9. Define como 1 se o desconto foi maior que 10% do preço base
df_registro['is_promo'] = (df_registro['diferenca_preco'] / df_registro['preco_base'] > 0.10).astype(int)

### 10. Inteligência Sazonal

O que faz: Cria uma flag binária para compras com desconto relevante (>10%).

Por que é importante: Categoriza os clientes pelo gatilho de compra. Alguns clientes buscam qualidade/marca, outros buscam apenas a oportunidade do preço baixo.

In [59]:
# 10. Extrai o mês para captar sazonalidade (Ex: Janeiro = Volta às Aulas)
df_registro['mes_venda'] = df_registro['data_venda'].dt.month


revisando data set

In [60]:
df_registro.head(1)

,data_venda,cliente_id,produto_id,qtd_vendida,preco_praticado,media_preco_similares,is_final_de_semana,estoque_disponivel,valor_total,nome_x,...,data_cadastro,estado,diferenca_preco,diferenca_preco_percentual,idade_cliente,tempo_cadastro_venda,dias_desde_ultima_compra,categoria_mais_comprada,is_promo,mes_venda
0,2026-03-01,CUST-1362,301,4,9.29,8.7,1,79,37.16,Refil Caneta Gel Pilot,...,2024-10-14,SC,-0.29,-0.032222,63,503,0.0,Escrita,0,3


In [61]:
datasets = {
    "Vendas": df_registro, 

}
for nome, df in datasets.items():
    nulos = df.isnull().sum().sum()
    duplicatas = df.duplicated().sum()
    print(f"{nome:12} | Nulos: {nulos} | Duplicatas: {duplicatas}")

Vendas       | Nulos: 0 | Duplicatas: 0


## Seleção de Atributos (Feature Selection)

Nesta etapa, apliquei a Seleção de Atributos (Feature Selection) para filtrar apenas as variáveis com alto poder preditivo e remover ruídos (como IDs e textos descritivos). Isso garante que o modelo de Clustering seja mais eficiente, rápido e matematicamente preciso.

In [62]:
# Lista de colunas para remover
colunas_para_remover = [

    'nome_x', 
    'nome_y', 
    'categoria', 
    'subcategoria', 
    'marca', 
    'caracteristicas', 
    'data_cadastro', 
    'estado', 
    'preco_base',
    'qtd_vendida',
    'preco_praticado',
    'media_preco_similares',
    'is_final_de_semana',
    'produto_id'
    # Remova se não for usar One-Hot Encoding nela agora
]

# Criando o dataset de treino
df_pre_treino = df_registro.drop(columns=colunas_para_remover)

print(f"Dataset pronto! Colunas restantes: {df_pre_treino.columns.tolist()}")

Dataset pronto! Colunas restantes: ['data_venda', 'cliente_id', 'estoque_disponivel', 'valor_total', 'genero', 'data_nascimento', 'diferenca_preco', 'diferenca_preco_percentual', 'idade_cliente', 'tempo_cadastro_venda', 'dias_desde_ultima_compra', 'categoria_mais_comprada', 'is_promo', 'mes_venda']


In [63]:
df_pre_treino.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2673 entries, 0 to 2672
Data columns (total 14 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   data_venda                  2673 non-null   datetime64[ns]
 1   cliente_id                  2673 non-null   object        
 2   estoque_disponivel          2673 non-null   int64         
 3   valor_total                 2673 non-null   float64       
 4   genero                      2673 non-null   object        
 5   data_nascimento             2673 non-null   datetime64[ns]
 6   diferenca_preco             2673 non-null   float64       
 7   diferenca_preco_percentual  2673 non-null   float64       
 8   idade_cliente               2673 non-null   int64         
 9   tempo_cadastro_venda        2673 non-null   int64         
 10  dias_desde_ultima_compra    2673 non-null   float64       
 11  categoria_mais_comprada     2673 non-null   object      

In [64]:
df_pre_treino[df_pre_treino.cliente_id == 'CUST-1361']

,data_venda,cliente_id,estoque_disponivel,valor_total,genero,data_nascimento,diferenca_preco,diferenca_preco_percentual,idade_cliente,tempo_cadastro_venda,dias_desde_ultima_compra,categoria_mais_comprada,is_promo,mes_venda
44,2026-03-01,CUST-1361,33,114.66,M,1975-08-04,5.34,0.044500,50,1061,0.0,Papelaria,0,3
170,2026-03-05,CUST-1361,2,283.41,M,1975-08-04,-5.47,-0.061461,50,1065,4.0,Papelaria,0,3
518,2026-03-16,CUST-1361,45,5.00,M,1975-08-04,0.00,0.000000,50,1076,11.0,Papelaria,0,3
1000,2026-04-01,CUST-1361,93,2.56,M,1975-08-04,-0.06,-0.024000,50,1092,16.0,Papelaria,0,4
1181,2026-04-07,CUST-1361,88,478.80,M,1975-08-04,0.30,0.002500,50,1098,6.0,Papelaria,0,4
1316,2026-04-11,CUST-1361,26,363.84,M,1975-08-04,-1.96,-0.022022,50,1102,4.0,Papelaria,0,4
1976,2026-05-04,CUST-1361,18,253.83,M,1975-08-04,4.39,0.049326,50,1125,23.0,Papelaria,0,5


# Agregação de Dados e Construção do Perfil do Cliente (Customer Profiling)
## Criando o dataset de treino 
a partir do dataset de clientes agregando as informações de vendas.

In [65]:
df_clientes_header = df_clientes.copy()
df_clientes_header = df_clientes_header[['cliente_id', 'data_nascimento','data_cadastro']].reset_index(drop=True)    

In [66]:
df_clientes_header.head()

,cliente_id,data_nascimento,data_cadastro
0,CUST-1000,1965-11-06,2024-12-01
1,CUST-1001,1997-10-13,2024-01-08
2,CUST-1002,2006-01-28,2024-05-06
3,CUST-1003,1987-08-04,2024-04-04
4,CUST-1004,1959-11-21,2024-05-25


### 1. Agregação por Comportamento (O "Cérebro" do Perfil)

O que faz: Consolida centenas de linhas de vendas em apenas uma linha por cliente. Transforma transações isoladas em métricas comportamentais (Médias, Modas e Amplitudes).

Por que é importante: O algoritmo de Clustering (como o K-Means) não consegue aprender com "vendas soltas", ele precisa de "entidades". Aqui você deu personalidade ao dado: agora o sistema sabe quem é o cliente que só compra em promoção ou aquele que tem um mês de pico específico.

In [67]:
# Preparando as agregações no dataset de registros (df_pre_treino)
df_features = df_pre_treino.groupby('cliente_id').agg(
    
    avg_desconto=('diferenca_preco', 'mean'),
    
    tempo_atividade_dias=('data_venda', lambda x: (x.max() - x.min()).days),
    
    propensao_promo=('is_promo', 'mean'),
    
    perfil_categoria=('categoria_mais_comprada', lambda x: x.mode()[0]),
    
    mes_pico_compra=('mes_venda', lambda x: x.mode()[0]),
    
    valor_total=('valor_total', 'sum'),
    
    data_da_ultima_compra=('data_venda', 'max')

).reset_index()

# Renomeando para clareza
df_features #.columns = ['cliente_id', 'avg_desconto', 'tempo_atividade_dias', 'propensao_promo', 'perfil_categoria', 'mes_pico_compra', 'valor_total', 'data_da_ultima_compra']

,cliente_id,avg_desconto,tempo_atividade_dias,propensao_promo,perfil_categoria,mes_pico_compra,valor_total,data_da_ultima_compra
0,CUST-1000,6.314286e-01,73,0.0,Escrita,4,500.67,2026-05-21
1,CUST-1001,-2.300000e-01,79,0.0,Escrita,3,700.57,2026-05-19
2,CUST-1002,-1.855000e+00,68,0.0,Escrita,5,246.12,2026-05-14
3,CUST-1003,-2.690000e+00,66,0.0,Escrita,3,619.02,2026-05-18
4,CUST-1004,8.200000e-01,71,0.0,Escrita,5,454.61,2026-05-16
...,...,...,...,...,...,...,...,...
494,CUST-1495,-5.366667e-01,78,0.0,Escrita,5,365.81,2026-05-27
495,CUST-1496,-7.771429e-01,71,0.0,Escrita,4,722.26,2026-05-23
496,CUST-1497,-9.200000e-02,73,0.0,Escrita,3,90.36,2026-05-19
497,CUST-1498,-8.428571e-02,77,0.0,Escrita,5,371.30,2026-05-23


In [68]:
df_pre_treino.head(1)

,data_venda,cliente_id,estoque_disponivel,valor_total,genero,data_nascimento,diferenca_preco,diferenca_preco_percentual,idade_cliente,tempo_cadastro_venda,dias_desde_ultima_compra,categoria_mais_comprada,is_promo,mes_venda
0,2026-03-01,CUST-1362,79,37.16,M,1962-12-24,-0.29,-0.032222,63,503,0.0,Escrita,0,3


In [69]:
df_transcoes_limp = df_vendas.copy()
df_transcoes_limp = df_transcoes_limp.groupby('cliente_id').size().reset_index(name='qtde_compras')

df_transcoes_limp.head(3)

,cliente_id,qtde_compras
0,CUST-1000,7
1,CUST-1001,8
2,CUST-1002,6


In [70]:
df_transacoes[df_transacoes['cliente_id'] == 'CUST-1280']

,id_transacao,cliente_id,produto_id,nome
2,T1002,CUST-1280,101,Caneta Gel Pilot G2 Preta
3,T1002,CUST-1280,301,Refil Caneta Gel Pilot
24,T1019,CUST-1280,101,Caneta Gel Pilot G2 Preta
25,T1019,CUST-1280,301,Refil Caneta Gel Pilot


### 2. Garantia de Integridade da Base (Left Join)

O que faz: Une o histórico de compras com a lista completa de clientes cadastrados.

Por que é importante: Em um cenário real, é vital saber quem são os clientes inativos. Ao usar o how='left', você garante que o modelo também enxergue os 71 clientes que nunca compraram (500 totais - 429 ativos), permitindo criar um cluster específico para "Novos Clientes" ou "Clientes sem Histórico".

In [71]:
# Unindo ao cabeçalho principal (Garantindo os 500 clientes)
df_clientes_header = pd.merge(df_clientes_header, df_features, on='cliente_id', how='left')


# V6: Idade atualizada (baseada no cadastro)
df_clientes_header['idade'] = 2026 - pd.to_datetime(df_clientes_header['data_nascimento']).dt.year

# V7: Tempo desde o cadastro (Tenure em dias)
data_hoje = pd.to_datetime('2026-04-06')
df_clientes_header['tenure_dias'] = (data_hoje - pd.to_datetime(df_clientes_header['data_cadastro'])).dt.days

# V8: Frequência total de compras (Contagem de transações)
df_clientes_header = pd.merge(df_clientes_header, df_transcoes_limp[['cliente_id', 'qtde_compras']], on='cliente_id', how='left')
df_clientes_header['qtde_compras'] = df_clientes_header['qtde_compras'].fillna(0).astype(int)

### 3. Tratamento de Dados Ausentes (Imputação de Valores)

O que faz: Substitui os valores nulos (NaN) gerados pelos clientes sem compras por valores matematicamente neutros ou rótulos descritivos.

Por que é importante: Algoritmos de Machine Learning falham na presença de valores nulos. Ao imputar 0 no desconto ou no tempo de atividade, você está dizendo ao modelo: "este indivíduo ainda não gerou valor/experiência", o que é uma informação valiosa para o agrupamento.

In [72]:
# Tratamento de nulos para quem não tem histórico de compra
df_clientes_header['avg_desconto'] = df_clientes_header['avg_desconto'].fillna(0)
df_clientes_header['tempo_atividade_dias'] = df_clientes_header['tempo_atividade_dias'].fillna(0)
df_clientes_header['propensao_promo'] = df_clientes_header['propensao_promo'].fillna(0)
df_clientes_header['perfil_categoria'] = df_clientes_header['perfil_categoria'].fillna('Sem_istórico')


### 4. Extração de Variáveis Demográficas e de Fidelidade (Tenure)

O que faz: Transforma datas estáticas (data_nascimento, data_cadastro) em números dinâmicos e comparáveis.

Por que é importante: * Idade: Ajuda a entender se o público de papelaria é majoritariamente jovem (estudantes) ou mais velho (uso profissional).

Tenure (Tempo de Casa): É a métrica definitiva de Lealdade. Um cliente com alto tenure_dias mas baixo tempo_atividade_dias é um cliente antigo que parou de comprar, um alvo perfeito para campanhas de reativação.

In [73]:

# Idade atualizada (baseada no cadastro)
df_clientes_header['idade'] = 2026 - pd.to_datetime(df_clientes_header['data_nascimento']).dt.year

# Tempo desde o cadastro (Tenure em dias)
data_hoje = pd.to_datetime('2026-04-06')
df_clientes_header['tenure_dias'] = (data_hoje - pd.to_datetime(df_clientes_header['data_cadastro'])).dt.days


df_clientes_header.head()

,cliente_id,data_nascimento,data_cadastro,avg_desconto,tempo_atividade_dias,propensao_promo,perfil_categoria,mes_pico_compra,valor_total,data_da_ultima_compra,idade,tenure_dias,qtde_compras
0,CUST-1000,1965-11-06,2024-12-01,0.631429,73.0,0.0,Escrita,4.0,500.67,2026-05-21,61,491,7
1,CUST-1001,1997-10-13,2024-01-08,-0.230000,79.0,0.0,Escrita,3.0,700.57,2026-05-19,29,819,8
2,CUST-1002,2006-01-28,2024-05-06,-1.855000,68.0,0.0,Escrita,5.0,246.12,2026-05-14,20,700,6
3,CUST-1003,1987-08-04,2024-04-04,-2.690000,66.0,0.0,Escrita,3.0,619.02,2026-05-18,39,732,3
4,CUST-1004,1959-11-21,2024-05-25,0.820000,71.0,0.0,Escrita,5.0,454.61,2026-05-16,67,681,6


### Codificação de Variáveis Categóricas (One-Hot Encoding)

O pd.get_dummies pega uma coluna de texto (como perfil_categoria, que contém "Escrita", "Papelaria", etc.) e a "explode" em múltiplas colunas binárias. Se um cliente pertence à categoria "Escrita", a coluna cat_Escrita recebe o valor 1 e todas as outras colunas de categoria para esse cliente recebem 0

In [74]:
df_clientes_header = pd.get_dummies(df_clientes_header, columns=['perfil_categoria'], prefix='cat')
print(df_clientes_header.filter(like='cat_').head())

   cat_Escrita  cat_Papelaria  cat_Sem_istórico
0         True          False             False
1         True          False             False
2         True          False             False
3         True          False             False
4         True          False             False


 removendo colunas de data e texto

In [75]:
df_clientes_header = df_clientes_header.drop(columns=['data_nascimento', 'data_cadastro'])

In [77]:
df_clientes_header.head()

,cliente_id,avg_desconto,tempo_atividade_dias,propensao_promo,mes_pico_compra,valor_total,data_da_ultima_compra,idade,tenure_dias,qtde_compras,cat_Escrita,cat_Papelaria,cat_Sem_istórico
0,CUST-1000,0.631429,73.0,0.0,4.0,500.67,2026-05-21,61,491,7,True,False,False
1,CUST-1001,-0.230000,79.0,0.0,3.0,700.57,2026-05-19,29,819,8,True,False,False
2,CUST-1002,-1.855000,68.0,0.0,5.0,246.12,2026-05-14,20,700,6,True,False,False
3,CUST-1003,-2.690000,66.0,0.0,3.0,619.02,2026-05-18,39,732,3,True,False,False
4,CUST-1004,0.820000,71.0,0.0,5.0,454.61,2026-05-16,67,681,6,True,False,False


### Criando o dataset final para modelagem

In [76]:
df_clientes_header.to_csv('../data/df_clientes_header.csv', index=False)

df_clientes_header.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   cliente_id             500 non-null    object        
 1   avg_desconto           500 non-null    float64       
 2   tempo_atividade_dias   500 non-null    float64       
 3   propensao_promo        500 non-null    float64       
 4   mes_pico_compra        499 non-null    float64       
 5   valor_total            499 non-null    float64       
 6   data_da_ultima_compra  499 non-null    datetime64[ns]
 7   idade                  500 non-null    int32         
 8   tenure_dias            500 non-null    int64         
 9   qtde_compras           500 non-null    int64         
 10  cat_Escrita            500 non-null    bool          
 11  cat_Papelaria          500 non-null    bool          
 12  cat_Sem_istórico       500 non-null    bool          
dtypes: bo

### Próximos Passos
Com os dados limpos e enriquecidos, o próximo passo é realizar a **Análise RFM** no `Notebook 02.ipynb`.